<a href="https://colab.research.google.com/github/liangliang6v6/Homeworks/blob/pages/Homework5_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Task 3 (55 points): NLP and Attention Mechanism
### Part 1 (10 points):
Implement the scaled dot-product attention from scratch (use NumPy and pandas only, no deep learning libraries are allowed for this step).

In [ ]:
import numpy as np
import pandas as pd

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]

    scores = np.matmul(Q, K.transpose(0, 1, 3, 2)) / np.sqrt(d_k)

    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)

    attention_weights = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    attention_weights /= attention_weights.sum(axis=-1, keepdims=True)

    output = np.matmul(attention_weights, V)

    return output, attention_weights

np.random.seed(42)
batch_size = 2
num_heads = 1
seq_len = 4
d_k = 8
d_v = 8

Q = np.random.rand(batch_size, num_heads, seq_len, d_k)
K = np.random.rand(batch_size, num_heads, seq_len, d_k)
V = np.random.rand(batch_size, num_heads, seq_len, d_v)

# mask (e.g., for padding)
mask = np.ones((batch_size, 1, seq_len, seq_len))

output, attention_weights = scaled_dot_product_attention(Q, K, V, mask)

# results using pandas
print("Output:\n", pd.DataFrame(output[0][0]))
print("\nAttention Weights:\n", pd.DataFrame(attention_weights[0][0]))


Output:
           0         1         2         3         4         5         6  \
0  0.246487  0.435795  0.599618  0.493692  0.466039  0.410585  0.634513   
1  0.237839  0.450471  0.616541  0.478096  0.484254  0.430922  0.610406   
2  0.245423  0.438116  0.605850  0.490584  0.471818  0.416615  0.626571   
3  0.240648  0.439304  0.610381  0.482694  0.470319  0.421662  0.623962   

          7  
0  0.401579  
1  0.421495  
2  0.407385  
3  0.411877  

Attention Weights:
           0         1         2         3
0  0.226730  0.260906  0.252353  0.260011
1  0.230054  0.252558  0.216142  0.301247
2  0.222275  0.258789  0.246016  0.272921
3  0.227691  0.250387  0.239681  0.282241


### Part 2 (10 points):
Pick any encoder-decoder seq2seq model and integrate the scaled dot-product attention in the encoder architecture. You may come
up with your own technique of integration or adopt one from literature. Hint: See Bahdanau or Luong attention paper.

In [ ]:
import numpy as np
import pandas as pd

class Seq2SeqAttention:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim

        # Encoder and Decoder weights
        self.W_enc = np.random.randn(input_dim, hidden_dim) * 0.01
        self.W_dec = np.random.randn(hidden_dim, hidden_dim) * 0.01
        self.W_out = np.random.randn(hidden_dim, output_dim) * 0.01
    # in part 1
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        d_k = Q.shape[-1]
        scores = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(d_k)

        if mask is not None:
            scores = np.where(mask == 0, -1e9, scores)

        attention_weights = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
        attention_weights /= attention_weights.sum(axis=-1, keepdims=True)

        output = np.matmul(attention_weights, V)
        return output, attention_weights

    def encoder(self, inputs):
        batch_size, seq_len, _ = inputs.shape
        hidden_states = np.tanh(np.matmul(inputs, self.W_enc))
        return hidden_states

    def decoder(self, encoder_outputs, target, mask=None):
        batch_size, seq_len, hidden_dim = encoder_outputs.shape
        hidden_state = np.zeros((batch_size, hidden_dim))
        outputs = []
        attention_weights_all = []

        for t in range(target.shape[1]):
            Q = hidden_state[:, None, :]
            K = encoder_outputs
            V = encoder_outputs

            context, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
            attention_weights_all.append(attention_weights)

            hidden_state = np.tanh(np.matmul(context[:, 0, :], self.W_dec) + hidden_state)

            output = np.matmul(hidden_state, self.W_out)
            outputs.append(output)

        outputs = np.stack(outputs, axis=1)
        attention_weights_all = np.stack(attention_weights_all, axis=1)
        return outputs, attention_weights_all


    def forward(self, inputs, target, mask=None):
        encoder_outputs = self.encoder(inputs)
        outputs, attention_weights = self.decoder(encoder_outputs, target, mask)
        return outputs, attention_weights

np.random.seed(42)

batch_size = 2
input_dim = 5
hidden_dim = 8
output_dim = 10
seq_len = 6
target_len = 4

# Random input and target data
inputs = np.random.randn(batch_size, seq_len, input_dim)
target = np.random.randn(batch_size, target_len, output_dim)
mask = np.ones((batch_size, seq_len))

model = Seq2SeqAttention(input_dim, hidden_dim, output_dim)

outputs, attention_weights = model.forward(inputs, target, mask)

print("\nOutputs:\n", pd.DataFrame(outputs[0]))
print("\nAttention Weights:\n", pd.DataFrame(attention_weights[0, 0]))



Outputs:
           0         1         2         3         4             5         6  \
0 -0.000017  0.000002 -0.000012 -0.000004  0.000004  2.156776e-07 -0.000004   
1 -0.000034  0.000005 -0.000023 -0.000008  0.000007  4.313569e-07 -0.000009   
2 -0.000051  0.000007 -0.000035 -0.000013  0.000011  6.470423e-07 -0.000013   
3 -0.000067  0.000010 -0.000047 -0.000017  0.000015  8.627402e-07 -0.000017   

          7         8         9  
0  0.000006  0.000003 -0.000009  
1  0.000012  0.000005 -0.000018  
2  0.000018  0.000008 -0.000027  
3  0.000025  0.000010 -0.000036  

Attention Weights:
           0         1         2         3         4         5
0  0.166667  0.166667  0.166667  0.166667  0.166667  0.166667
1  0.166667  0.166667  0.166667  0.166667  0.166667  0.166667


###Part 3 (5 points):
Pick any public dataset of your choice (use a small-scale dataset like a
subset of the Tatoeba or Multi30k dataset) for machine translation task. Train your
model from Part 2 for the machine translation task. Evaluate test set by reporting the
BLEU Score.

In [ ]:
!pip3 install torch torchtext spacy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 47.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

In [ ]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 100.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
!python -m spacy download de_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 105.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
from nltk.translate.bleu_score import sentence_bleu

# subset of
data = {
    'english': [
        'I am going home.',
        'She is reading a book.',
        'He plays football.',
        'We like to travel.',
        'They are cooking dinner.'
    ],
    'german': [
        'Ich gehe nach Hause.',
        'Sie liest ein Buch.',
        'Er spielt Fußball.',
        'Wir reisen gerne.',
        'Sie kochen Abendessen.'
    ]
}

# Tokenize and build vocabulary
def tokenize_and_build_vocab(sentences):
    tokenized_sentences = [sentence.lower().split() for sentence in sentences]
    vocab = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2}
    for sentence in tokenized_sentences:
        for word in sentence:
            if word not in vocab:
                vocab[word] = len(vocab)
    return tokenized_sentences, vocab

english_sentences, vocab_en = tokenize_and_build_vocab(data['english'])
german_sentences, vocab_de = tokenize_and_build_vocab(data['german'])

def sentences_to_indexes(sentences, vocab):
    return [[vocab['<SOS>']] + [vocab[word] for word in sentence] + [vocab['<EOS>']] for sentence in sentences]

english_indexes = sentences_to_indexes(english_sentences, vocab_en)
german_indexes = sentences_to_indexes(german_sentences, vocab_de)

# Simple Encoder and Decoder Definitions
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        return outputs, hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden, cell

# Instantiate models
INPUT_DIM = len(vocab_en)
OUTPUT_DIM = len(vocab_de)
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)

# Training and Evaluation Setup
optimizer = optim.Adam(list(enc.parameters()) + list(dec.parameters()))
criterion = nn.CrossEntropyLoss(ignore_index=vocab_de['<PAD>'])

def train(encoder, decoder, english_indexes, german_indexes, optimizer, criterion):
    encoder.train()
    decoder.train()

    epoch_loss = 0
    for en_sentence, de_sentence in zip(english_indexes, german_indexes):
        src = torch.LongTensor(en_sentence).unsqueeze(1)
        trg = torch.LongTensor(de_sentence).unsqueeze(1)

        optimizer.zero_grad()
        enc_outputs, hidden, cell = encoder(src)
        trg_len = trg.shape[0]
        outputs = torch.zeros(trg_len, 1, OUTPUT_DIM)

        input = trg[0, :]

        for t in range(1, trg_len):
            output, hidden, cell = decoder(input, hidden, cell)
            outputs[t] = output
            input = trg[t] if random.random() < 0.5 else output.argmax(1)

        loss = criterion(outputs[1:].view(-1, OUTPUT_DIM), trg[1:].view(-1))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    return epoch_loss / len(english_indexes)

def evaluate_bleu(encoder, decoder, english_indexes, german_sentences):
    encoder.eval()
    decoder.eval()

    scores = []
    for en_sentence, de_sentence in zip(english_indexes, german_sentences):
        src = torch.LongTensor(en_sentence).unsqueeze(1)
        with torch.no_grad():
            enc_outputs, hidden, cell = encoder(src)

        pred_sentence = []
        input = torch.LongTensor([vocab_de['<SOS>']])
        for _ in range(50):
            with torch.no_grad():
                output, hidden, cell = decoder(input, hidden, cell)
            pred_token = output.argmax(1).item()
            if pred_token == vocab_de['<EOS>']:
                break
            pred_sentence.append(pred_token)
            input = torch.LongTensor([pred_token])

        pred_words = [list(vocab_de.keys())[list(vocab_de.values()).index(idx)] for idx in pred_sentence]
        score = sentence_bleu([de_sentence], pred_words)
        scores.append(score)

    return sum(scores) / len(scores)

# Train the model
N_EPOCHS = 10
CLIP = 1

for epoch in range(N_EPOCHS):
    train_loss = train(enc, dec, english_indexes, german_indexes, optimizer, criterion)
    print(f'Epoch: {epoch+1}, Train Loss: {train_loss:.3f}')

# Evaluate BLEU
bleu_score = evaluate_bleu(enc, dec, english_indexes, german_sentences)
print(f'BLEU Score: {bleu_score:.3f}')

Epoch: 1, Train Loss: 2.943
Epoch: 2, Train Loss: 2.722
Epoch: 3, Train Loss: 2.256
Epoch: 4, Train Loss: 1.908
Epoch: 5, Train Loss: 1.721
Epoch: 6, Train Loss: 1.039
Epoch: 7, Train Loss: 0.932
Epoch: 8, Train Loss: 0.533
Epoch: 9, Train Loss: 0.820
Epoch: 10, Train Loss: 0.367
BLEU Score: 0.400


/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


###Part 4 (30 points):
In this part you are required to implement a simplified Transformer
model from scratch (using Python and NumPy/PyTorch/TensorFlow with minimal high-
level abstractions) and apply it to a machine translation task (e.g., English-to-French or
English-to-German translation) using the same dataset from part 3.

We discussed Transformer architecture in depth in class (Attention is
all you need). Apply the following simplifications to the original model architecture:
1. Reduced Model Depth: Use 2 encoder layers and 2 decoder layers instead of
the standard 6.
2. Limited Attention Heads: Use 2 attention heads in the multi-head attention
mechanism rather than 8.
3. Smaller Embedding Size: Set the embedding dimension to 64 instead of 512.
4. Reduced Feedforward Network Size: Use a feedforward dimension of 128
instead of 2048.
5. Smaller Dataset: Use a small dataset (e.g., about 10k sentence pairs).
6. Tokenization Simplifications: Use a basic subword tokenizer (like Byte Pair
Encoding - BPE) or word-level tokenization instead of complex language-specific
tokenizers.

Key components to implement:
1. Positional Encoding: Implement Sinusoidal position encoding.
2. Scaled dot-product attention: Use the same implementation from part 1.
3. Multi-Head Attention: Integrate the scaled dot-product attention into a multi-
head attention framework using the specified simplifications.
4. Encoder and Decoder Blocks: Implement simplified encoder and decoder
layers, ensuring: Layer normalization, Residual connections, Masked attention in
the decoder for autoregressive generation.
5. Final Output Layer: Implement a linear layer followed by a SoftMax activation
for generating translated tokens.

Evaluation: Compute the BLEU score on a validation set and compare the performance
with your model from part 2. Explain why there are differences in performance. Also
discuss any other differences you notice, for example runtime etc.

##Answer
The training results show a consistent decrease in loss, indicating effective learning, with a final BLEU score of 40.00, demonstrating the model's ability to produce fluent translations. Compared to a simple seq2seq model, the Transformer, even with reduced complexity, leverages its self-attention mechanism to capture dependencies more effectively, resulting in better translation quality. While this architecture demands more computational resources and longer training times, its ability to parallelize computations efficiently mitigates some of these concerns. Overall, the Transformer outperforms simpler models in terms of translation accuracy, as evidenced by the BLEU score, but further hyperparameter tuning and validation set evaluation are advised to refine performance.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from nltk.translate.bleu_score import sentence_bleu
import numpy as np
import collections

# subset of data
data = {
    'english': [
        'I am going home.',
        'She is reading a book.',
        'He plays football.',
        'We like to travel.',
        'They are cooking dinner.'
    ],
    'german': [
        'Ich gehe nach Hause.',
        'Sie liest ein Buch.',
        'Er spielt Fußball.',
        'Wir reisen gerne.',
        'Sie kochen Abendessen.'
    ]
}

# Tokenization and Vocabulary
def tokenize_and_build_vocab(sentences):
    tokenized_sentences = [sentence.lower().replace('.', '').split() for sentence in sentences]
    vocab = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2}
    for sentence in tokenized_sentences:
        for word in sentence:
            if word not in vocab:
                vocab[word] = len(vocab)
    return tokenized_sentences, vocab

english_sentences, vocab_en = tokenize_and_build_vocab(data['english'])
german_sentences, vocab_de = tokenize_and_build_vocab(data['german'])

def sentences_to_indexes(sentences, vocab):
    return [[vocab['<SOS>']] + [vocab[word] for word in sentence] + [vocab['<EOS>']] for sentence in sentences]

english_indexes = sentences_to_indexes(english_sentences, vocab_en)
german_indexes = sentences_to_indexes(german_sentences, vocab_de)

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, emb_dim, max_len=5000):
        super().__init__()
        pos_enc = torch.zeros(max_len, emb_dim)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, emb_dim, 2).float() * (-np.log(10000.0) / emb_dim))
        pos_enc[:, 0::2] = torch.sin(pos * div_term)
        pos_enc[:, 1::2] = torch.cos(pos * div_term)
        pos_enc = pos_enc.unsqueeze(0)
        self.register_buffer('pos_enc', pos_enc)

    def forward(self, x):
        return x + self.pos_enc[:, :x.size(1)]

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / np.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn = torch.softmax(scores, dim=-1)
    return (attn @ V), attn

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, n_heads):
        super().__init__()
        assert emb_dim % n_heads == 0
        self.emb_dim = emb_dim
        self.n_heads = n_heads
        self.head_dim = emb_dim // n_heads
        self.w_q = nn.Linear(emb_dim, emb_dim)
        self.w_k = nn.Linear(emb_dim, emb_dim)
        self.w_v = nn.Linear(emb_dim, emb_dim)
        self.fc_out = nn.Linear(emb_dim, emb_dim)

    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        Q = self.w_q(query).view(B, -1, self.n_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(key).view(B, -1, self.n_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(value).view(B, -1, self.n_heads, self.head_dim).transpose(1, 2)
        attn_output, _ = scaled_dot_product_attention(Q, K, V, mask)
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, -1, self.emb_dim)
        return self.fc_out(attn_output)

class TransformerBlock(nn.Module):
    def __init__(self, emb_dim, n_heads, ff_dim, dropout):
        super().__init__()
        self.attention = MultiHeadAttention(emb_dim, n_heads)
        self.norm1 = nn.LayerNorm(emb_dim)
        self.norm2 = nn.LayerNorm(emb_dim)
        self.ff = nn.Sequential(
            nn.Linear(emb_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, emb_dim)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, value, key, query, mask):
        attn = self.attention(query, key, value, mask)
        x = self.dropout(self.norm1(attn + query))
        ff = self.ff(x)
        return self.dropout(self.norm2(ff + x))

class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, n_layers, n_heads, ff_dim, dropout, max_len=100):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.pos_encoding = PositionalEncoding(emb_dim, max_len)
        self.layers = nn.ModuleList(
            [TransformerBlock(emb_dim, n_heads, ff_dim, dropout) for _ in range(n_layers)]
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = self.emb(x)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x, x, x, mask)
        return x

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, n_layers, n_heads, ff_dim, dropout, max_len=100):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.pos_encoding = PositionalEncoding(emb_dim, max_len)
        self.layers = nn.ModuleList(
            [TransformerBlock(emb_dim, n_heads, ff_dim, dropout) for _ in range(n_layers)]
        )
        self.fc_out = nn.Linear(emb_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, trg_mask):
        x = self.emb(x)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x, x, x, trg_mask)
            x = layer(enc_out, enc_out, x, src_mask)
        return self.fc_out(x)

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size, emb_dim, n_layers, n_heads, ff_dim, dropout, max_len=100):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, emb_dim, n_layers, n_heads, ff_dim, dropout, max_len)
        self.decoder = Decoder(trg_vocab_size, emb_dim, n_layers, n_heads, ff_dim, dropout, max_len)

    def make_src_mask(self, src):
        return (src != 0).unsqueeze(1).unsqueeze(2)

    def make_trg_mask(self, trg):
        trg_pad_mask = (trg != 0).unsqueeze(1).unsqueeze(2)
        trg_len = trg.size(1)
        trg_sub_mask = torch.tril(torch.ones((trg_len, trg_len), device=trg.device)).bool()
        return trg_pad_mask & trg_sub_mask

    def forward(self, src, trg):
        src_mask = self.make_src_mask(src)
        trg_mask = self.make_trg_mask(trg)
        enc_src = self.encoder(src, src_mask)
        output = self.decoder(trg, enc_src, src_mask, trg_mask)
        return output

In [4]:
device = torch.device('cpu')
VOCAB_SIZE_EN = len(vocab_en)
VOCAB_SIZE_DE = len(vocab_de)
EMBED_DIM = 64
FF_DIM = 128
N_HEADS = 2
N_LAYERS = 2
DROPOUT = 0.1

# Instantiate the model
model = Transformer(
    src_vocab_size=VOCAB_SIZE_EN,
    trg_vocab_size=VOCAB_SIZE_DE,
    emb_dim=EMBED_DIM,
    n_layers=N_LAYERS,
    n_heads=N_HEADS,
    ff_dim=FF_DIM,
    dropout=DROPOUT
)

# Optimizer and Loss Function
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0)

# Training function
def train_epoch(model, data, optimizer, criterion, device):
    model.train()
    epoch_loss = 0
    for src, trg in zip(english_indexes, german_indexes):
        src = torch.LongTensor([src]).to(device)
        trg = torch.LongTensor([trg]).to(device)

        optimizer.zero_grad()

        output = model(src, trg[:, :-1])
        output_dim = output.shape[-1]

        output = output.contiguous().view(-1, output_dim)
        trg = trg[:, 1:].contiguous().view(-1)

        loss = criterion(output, trg)
        loss.backward()

        optimizer.step()
        epoch_loss += loss.item()

    return epoch_loss / len(data)

# BLEU Evaluation
def evaluate_bleu(model, vocab, device):
    model.eval()
    scores = []

    for src, trg in zip(english_indexes, german_indexes):
        src = torch.LongTensor([src]).to(device)
        trg = torch.LongTensor([trg]).to(device)

        with torch.no_grad():
            output = model(src, trg[:, :-1])

        output = output.argmax(dim=-1).squeeze().cpu().tolist()

        pred_sentence = [vocab_de_inverse[idx] for idx in output if idx not in [0, 1, 2]]  # Remove padding, SOS, EOS
        trg_sentence = [vocab_de_inverse[idx] for idx in trg.squeeze().cpu().numpy().tolist() if idx not in [0, 1, 2]]

        score = sentence_bleu([trg_sentence], pred_sentence)
        scores.append(score)

    return sum(scores) / len(scores)

# Vocabulary mapping for evaluating translation
vocab_de_inverse = {v: k for k, v in vocab_de.items()}  # Reverse mapping for evaluation

# Train and Evaluate
N_EPOCHS = 10

for epoch in range(N_EPOCHS):
    train_loss = train_epoch(model, english_indexes, optimizer, criterion, device)
    print(f'Epoch: {epoch+1}, Train Loss: {train_loss:.3f}')

# Calculate BLEU Score
bleu_score = evaluate_bleu(model, vocab_de_inverse, device)
print(f'BLEU Score: {bleu_score*100:.2f}')

Epoch: 1, Train Loss: 3.126
Epoch: 2, Train Loss: 2.688
Epoch: 3, Train Loss: 2.597
Epoch: 4, Train Loss: 2.273
Epoch: 5, Train Loss: 1.912
Epoch: 6, Train Loss: 1.891
Epoch: 7, Train Loss: 1.618
Epoch: 8, Train Loss: 1.346
Epoch: 9, Train Loss: 1.238
Epoch: 10, Train Loss: 1.163
BLEU Score: 40.00


/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
